LAVAGRID ENVIRONMENT SETUP

In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
from minigrid.wrappers import FullyObsWrapper, RGBImgPartialObsWrapper

In [2]:
env = FullyObsWrapper(gym.make("MiniGrid-LavaCrossingS11N5-v0"))
obs, _ = env.reset()
obs['image'].shape

(11, 11, 3)

In [3]:
env_obs = FullyObsWrapper(env)
obs, _ = env_obs.reset()
obs['image'].shape

(11, 11, 3)

In [4]:
def policy(observation):
    return env.action_space.sample()

env = gym.make("MiniGrid-LavaGapS7-v0", render_mode="human")
env = FullyObsWrapper(env)
# env = RGBImgPartialObsWrapper(env)
observation, info = env.reset(seed=42)
print(observation['image'].shape)
for _ in range(1):
    action = policy(observation)  # User-defined policy function
    observation, reward, terminated, truncated, info = env.step(action)
    print(observation)
    if terminated or truncated:
        observation, info = env.reset()
env.close()

(7, 7, 3)
{'image': array([[[ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [10,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 9,  0,  0],
        [ 9,  0,  0],
        [ 9,  0,  0],
        [ 1,  0,  0],
        [ 9,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 1,  0,  0],
        [ 8,  1,  0],
        [ 2,  5,  0]],

       [[ 2,  5,  0],
        [ 2,  5,  0],


---

## Prioritized Replay Buffer

In [5]:
from collections import deque

class PrioritizedReplayBuffer:
    def __init__(
        self, capacity, alpha=0.6, beta=0.4, beta_increment=0.001, epsilon=0.01
    ):
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)
        self.alpha, self.beta, self.beta_increment, self.epsilon = (alpha, beta, beta_increment, epsilon)
        self.priorities = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        experience_tuple = (state, action, reward, next_state, done)
        # Initialize the transition's priority
        max_priority = int(self.capacity) # MODIFY
        self.memory.append(experience_tuple)
        self.priorities.append(max_priority)
    
    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors.tolist()):
            # Update the transition's priority
            self.priorities[idx] = idx # MODIFY

    def increase_beta(self):
        # Increase beta if less than 1
        self.beta = 1 # MODIFY

    def __len__(self):
        return len(self.memory)


Test the Replay Buffer

In [6]:
buffer = PrioritizedReplayBuffer(capacity=3)
buffer.push(state=[1,3], action=2, reward=1, next_state=[2,4], done=False)


In [7]:
print("Transition in memory buffer:", buffer.memory)
print("Priority buffer:", buffer.priorities)

Transition in memory buffer: deque([([1, 3], 2, 1, [2, 4], False)], maxlen=3)
Priority buffer: deque([3], maxlen=3)


---

## Test PPO

In [ ]:
import os
import torch
import PPO as ppo

In [9]:
policy = ppo.PPO(lr=7e-4, adam_epsilon=1e-5)

In [10]:
policy.learn(n_updates=1000, n_steps=16384, gamma_=0.999, lambda_=0.95, clip_epsilon=0.2, minibatch=8, value_coeff=0.5, entropy_coeff=0.01)

[PPO] Update 1/1000 completed 	| Mean episode reward : 0.0174
[PPO] Update 2/1000 completed 	| Mean episode reward : 0.0256
[PPO] Update 3/1000 completed 	| Mean episode reward : 0.0311
[PPO] Update 4/1000 completed 	| Mean episode reward : 0.0495
[PPO] Update 5/1000 completed 	| Mean episode reward : 0.0538
[PPO] Update 6/1000 completed 	| Mean episode reward : 0.0612
[PPO] Update 7/1000 completed 	| Mean episode reward : 0.1099
[PPO] Update 8/1000 completed 	| Mean episode reward : 0.1796
[PPO] Update 9/1000 completed 	| Mean episode reward : 0.2189
[PPO] Update 10/1000 completed 	| Mean episode reward : 0.3095
[PPO] Update 11/1000 completed 	| Mean episode reward : 0.3239
[PPO] Update 12/1000 completed 	| Mean episode reward : 0.4209
[PPO] Update 13/1000 completed 	| Mean episode reward : 0.5650
[PPO] Update 14/1000 completed 	| Mean episode reward : 0.5899
[PPO] Update 15/1000 completed 	| Mean episode reward : 0.5824
[PPO] Update 16/1000 completed 	| Mean episode reward : 0.6576
[

KeyboardInterrupt: 

In [11]:
def evaluate_policy(agent, env_name="MiniGrid-LavaGapS7-v0", n_episodes=10, render=True):
        # Creiamo un ambiente per il test (magari con render_mode="human" se vuoi vedere)
        render_mode = "human" if render else None
        
        test_env = FullyObsWrapper(gym.make(env_name, render_mode=render_mode))
        agent.actor.eval() # Imposta la rete in modalità valutazione
        total_rewards = []

        for ep in range(n_episodes):
            obs, _ = test_env.reset()
            agent._reset_stack(obs['image'])
            done = False
            ep_reward = 0
            
            while not done:
                # Usiamo torch.no_grad per efficienza
                with torch.no_grad():
                    state_stacked = agent._get_stacked_obs(obs['image'])
                    dist = agent.actor(state_stacked)
                    # Scegliamo l'azione migliore (non randomica)
                    action = dist.probs.argmax(dim=-1).item()
                
                obs, reward, terminated, truncated, _ = test_env.step(action)
                done = terminated or truncated
                ep_reward += reward
                
            total_rewards.append(ep_reward)
            print(f"Episode {ep+1}: Reward = {ep_reward:.2f}")
        test_env.close()

In [16]:
policy.eval()
evaluate_policy(agent=policy)

Episode 1: Reward = 0.95
Episode 2: Reward = 0.95
Episode 3: Reward = 0.94
Episode 4: Reward = 0.95
Episode 5: Reward = 0.95
Episode 6: Reward = 0.95
Episode 7: Reward = 0.94
Episode 8: Reward = 0.94
Episode 9: Reward = 0.95
Episode 10: Reward = 0.95


In [ ]:
torch.save(policy.state_dict(), os.path.join(os.getcwd(), "simple_ppo.pt"))